In [17]:
from google.colab import drive
drive.mount('/content/drive')

data_dir = "/content/drive/MyDrive/ETHZ_ALL"


import pandas as pd
import glob
import os
import gc
import logging
from typing import Generator, List, Set
from multiprocessing import Pool, cpu_count
from functools import partial
from tqdm import tqdm
import pytz


import pytz

def detect_ev_charging_fast(customer_df,
                            step_threshold_kwh=1.5,
                            min_duration_minutes=60):
    """
    Highly optimized EV charging detection across ALL days.
    Includes same output format as your original function.
    """

    df = customer_df

    # Ensure timestamp index in UTC
    if df.index.tz is None:
        df.index = df.index.tz_localize("UTC")
    else:
        df.index = df.index.tz_convert("UTC")

    # Keep only required columns (faster operations)
    df = df[["CONSO_KWH", "PROD_KWH"]]

    # ---- 15-min resample ----
    df_15 = df.resample("15min").sum()

    # Precompute diff (fast)
    conso = df_15["CONSO_KWH"].fillna(0)
    df_15["diff"] = conso.diff()

    # Candidate starts/ends
    starts = df_15.index[df_15["diff"] > step_threshold_kwh]
    ends   = df_15.index[df_15["diff"] < -step_threshold_kwh]

    events = []
    ends = list(ends)
    ei = 0  # pointer into ends list
    n_ends = len(ends)

    # Precompute Zurich conversion once
    tz_ch = pytz.timezone("Europe/Zurich")
    local_idx = df_15.index.tz_convert(tz_ch)
    tz_map = dict(zip(df_15.index, local_idx))

    # ---- Scan candidates ----
    for s in starts:

        # Move pointer forward until ends[ei] > s
        while ei < n_ends and ends[ei] <= s:
            ei += 1

        if ei >= n_ends:
            break

        end_time = ends[ei]

        # Duration check
        duration_min = (end_time - s).total_seconds() / 60
        if duration_min < min_duration_minutes:
            continue

        # Interval check
        interval = conso.loc[s:end_time]
        if not (interval >= step_threshold_kwh).all():
            continue

        # Build event
        s_local = tz_map[s]
        e_local = tz_map[end_time]

        events.append({
            "start": s_local.strftime("%d.%m.%Y %H:%M"),
            "end":   e_local.strftime("%d.%m.%Y %H:%M"),
            "day":   s_local.strftime("%Y-%m-%d"),
            "duration_h": duration_min / 60,
            "consumption_start": conso.shift(1).loc[s],
            "consumption_trigger_start": conso.loc[s],
            "consumption_before_end": conso.loc[end_time - pd.Timedelta(minutes=15)],
            "consumption_trigger_end": conso.loc[end_time],
        })

    return pd.DataFrame(events), df_15


# --- Configure Logging --------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("data_loading.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# --- Worker function ----------------------------------------------------------
def process_single_file(file_path: str, valid_ids_set: Set[str], target_meta: pd.DataFrame) -> pd.DataFrame:
    try:
        df = pd.read_parquet(
            file_path,
            columns=["ID", "DT_UTC", "CONSO_KWH", "PROD_KWH"],
            engine='pyarrow'
        )

        if df.empty:
            return pd.DataFrame()

        df["ID"] = df["ID"].astype(str)
        filtered_df = df[df["ID"].isin(valid_ids_set)].copy()

        if filtered_df.empty:
            return pd.DataFrame()

        filtered_df["DT_UTC"] = pd.to_datetime(filtered_df["DT_UTC"])
        return filtered_df.merge(target_meta, on="ID", how="left")

    except Exception as e:
        logger.error(f"Failed to process {file_path}: {str(e)}")
        return pd.DataFrame()


# --- Parallel Loader ----------------------------------------------------------
def load_all_data_parallel_generator(
    data_dir: str,
    partner_type: str = "Particuliers",
    batch_size: int = 2
) -> Generator[pd.DataFrame, None, None]:

    logger.info(f"Starting data load for type: {partner_type}")

    # ✅ Load Metadata
    try:
        meta_df = pd.read_parquet(os.path.join(data_dir, "metadata"), engine='pyarrow')
        target_meta = meta_df[meta_df["TYPE_PARTENAIRE_LIBELLE"] == partner_type].copy()
        target_meta["ID"] = target_meta["ID"].astype(str)
        valid_ids_set = set(target_meta["ID"])
        logger.info(f"Metadata loaded. {len(valid_ids_set)} valid IDs.")
    except Exception as e:
        logger.critical(f"Could not load metadata: {e}")
        return

    # ✅ Find parquet files
    files = [f for f in glob.glob(os.path.join(data_dir, "*.parquet")) if "metadata" not in f.lower()]
    if not files:
        logger.error(f"No parquet files found in {data_dir}")
        return

    n_cores = max(1, cpu_count() // 2)
    batch_size = batch_size or n_cores

    worker_func = partial(process_single_file, valid_ids_set=valid_ids_set, target_meta=target_meta)

    with tqdm(total=len(files), desc="Overall Progress") as pbar:
        with Pool(processes=n_cores) as pool:
            for i in range(0, len(files), batch_size):
                file_chunk = files[i : i + batch_size]

                results = pool.map(worker_func, file_chunk)
                pbar.update(len(file_chunk))

                batch_df = pd.concat([res for res in results if not res.empty], ignore_index=True)
                if not batch_df.empty:
                    yield batch_df

                del results
                gc.collect()

    logger.info("✅ Data loading sequence complete.")


# =====================================================================
# ✅ MAIN EXECUTION BLOCK NEEDED FOR GOOGLE COLAB MULTIPROCESSING
# =====================================================================
if __name__ == "__main__":
    results = []

    data_gen = load_all_data_parallel_generator(
        data_dir=data_dir,
        partner_type="Particuliers",
        batch_size=4
    )

    for batch_df in data_gen:

        for cust_id, cust_df in batch_df.groupby("ID"):


            cust_df = cust_df.set_index("DT_UTC").sort_index()

            # ✅ Run optimized detection ONCE
            events, df15 = detect_ev_charging_fast(cust_df)

            results.append({
                "ID": cust_id,
                "n_events": len(events),
                "events": events.to_dict(orient="records"),
                "avg_conso": cust_df["CONSO_KWH"].mean()
            })



        del batch_df
        gc.collect()
        print("✅ One batch done!")

    final = pd.DataFrame(results)

    output_path = "/content/drive/MyDrive/ev_detection_results3.parquet"
    final.to_parquet(output_path)

    print("✅ Processing complete! Results saved to:", output_path)

    


ModuleNotFoundError: No module named 'google.colab'